# Airport Policy RAG Pipeline


## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

POLICY_DIR = Path("../data/airport_policies")

policy_files = list(POLICY_DIR.glob("*.md"))

print(f"Policy documents: {len(policy_files)}")

# Certificate issue resolve:
import os

os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

/home/nineleaps/Documents/da_python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Policy documents: 9


## 2. Load Policy Documents


In [2]:
documents = []

for file_path in policy_files:
    text = file_path.read_text(encoding="utf-8")

    documents.append({
        "source": file_path.name,
        "text": text
    })

print(documents[0]["source"])
print(documents[0]["text"][:500])

jfk_pricing.md
# JFK Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at JFK is 1.5x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.5x is prohibited.

## Surge Conditions

Surge may be considered when demand significantly exceeds available supply or when completion rate falls below the operational threshold.

All surge changes must include a documented operational reason.


## 3. Clean and Prepare Documents


In [3]:
def get_policy_metadata(filename):
    name = filename.lower()

    if "sfo" in name:
        airport = "SFO"
    elif "lax" in name:
        airport = "LAX"
    elif "jfk" in name:
        airport = "JFK"
    else:
        airport = "UNKNOWN"

    if "pricing" in name:
        policy_type = "pricing"
    elif "operations" in name:
        policy_type = "operations"
    elif "driver" in name:
        policy_type = "driver"
    else:
        policy_type = "unknown"

    return airport, policy_type

for doc in documents:
    airport, policy_type = get_policy_metadata(doc["source"])

    doc["airport"] = airport
    doc["policy_type"] = policy_type

documents[0]

{'source': 'jfk_pricing.md',
 'text': '# JFK Pricing Policy\n\n## Surge Pricing\n\nThe maximum permitted surge multiplier at JFK is 1.5x.\n\nSurge above 1.3x requires Operations Manager approval.\n\nSurge above 1.5x is prohibited.\n\n## Surge Conditions\n\nSurge may be considered when demand significantly exceeds available supply or when completion rate falls below the operational threshold.\n\nAll surge changes must include a documented operational reason.',
 'airport': 'JFK',
 'policy_type': 'pricing'}

## 4. Split Documents into Chunks


In [4]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [5]:
chunks = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"])

    for chunk in doc_chunks:
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "airport": doc["airport"],
            "policy_type": doc["policy_type"]
        })

print(f"Total chunks: {len(chunks)}")

Total chunks: 19


## 5. Generate Embeddings


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4721.51it/s]


In [7]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

print(embeddings.shape)

(19, 384)


## 6. Build FAISS Vector Index


In [8]:
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Indexed vectors: {index.ntotal}")

Indexed vectors: 19


## 7. Semantic Retrieval


In [9]:
def search_policy(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        result = chunks[idx].copy()
        result["score"] = float(score)
        results.append(result)

    return results

In [10]:
results = search_policy(
    "What is the maximum surge multiplier allowed at SFO?"
)

for result in results:
    print(result["source"])
    print(result["score"])
    print(result["text"])
    print()

sfo_pricing.md
0.8605122566223145
# SFO Pricing Policy

## Surge Pricing

The normal surge range is 1.0x to 1.5x.

The maximum permitted surge multiplier at SFO is 1.5x.

Surge increases above 1.3x require Operations Manager approval.

A surge multiplier above 1.5x is prohibited.

## Surge Conditions

Surge may be considered when request volume significantly exceeds available driver supply, completion rate falls below 85%, or airport queue conditions indicate insufficient supply.

Every surge adjustment must have a documented op

lax_pricing.md
0.7299853563308716
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surge Conditions
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surge Conditions

Surge may be consid

## 8. Build RAG Context


In [11]:
def build_context(results):
    # Combine the retrieved chunks into one context string for the LLM
    context = ""

    for result in results:
        # Include the source so the LLM knows where the information came from
        context += f"Source: {result['source']}\n"
        context += f"{result['text']}\n\n"

    return context

In [12]:
results = search_policy("What is the maximum surge multiplier allowed at SFO?")

# Convert the retrieved chunks into text that can be given to the LLM
context = build_context(results)

print(context)

Source: sfo_pricing.md
# SFO Pricing Policy

## Surge Pricing

The normal surge range is 1.0x to 1.5x.

The maximum permitted surge multiplier at SFO is 1.5x.

Surge increases above 1.3x require Operations Manager approval.

A surge multiplier above 1.5x is prohibited.

## Surge Conditions

Surge may be considered when request volume significantly exceeds available driver supply, completion rate falls below 85%, or airport queue conditions indicate insufficient supply.

Every surge adjustment must have a documented op

Source: lax_pricing.md
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surge Conditions
# LAX Pricing Policy

## Surge Pricing

The maximum permitted surge multiplier at LAX is 1.6x.

Surge above 1.3x requires Operations Manager approval.

Surge above 1.6x is prohibited.

## Surge Conditions

Surge may be considered when request volu

In [22]:
def build_rag_prompt(question, context):
    # Restrict the LLM to information found in retrieved policies
    prompt = f"""
You are an Airport Operations Policy Assistant.

Answer the user's question using ONLY the policy context provided below.

Rules:
- Do not invent policy rules, limits, approvals, or operational facts.
- If the answer is not available in the context, say:
  "I could not find this information in the available airport policies."
- Keep the answer concise.
- Mention the specific source document that supports the answer.

Policy Context:
{context}

User Question:
{question}

Return the response in exactly this format:

Answer: <concise answer>
Source: <policy filename>
"""

    return prompt

In [23]:
question = "What is the maximum surge multiplier allowed at SFO?"

# Retrieve the most relevant policy chunks for the question
results = search_policy(question)

# Convert retrieved chunks into LLM-readable context
context = build_context(results)

# Build the final grounded prompt
prompt = build_rag_prompt(question, context)

print(prompt)


You are an Airport Operations Policy Assistant.

Answer the user's question using ONLY the policy context provided below.

Rules:
- Do not invent policy rules, limits, approvals, or operational facts.
- If the answer is not available in the context, say:
  "I could not find this information in the available airport policies."
- Keep the answer concise.
- Mention the specific source document that supports the answer.

Policy Context:
Source: sfo_pricing.md
# SFO Pricing Policy

## Surge Pricing

The normal surge range is 1.0x to 1.5x.

The maximum permitted surge multiplier at SFO is 1.5x.

Surge increases above 1.3x require Operations Manager approval.

A surge multiplier above 1.5x is prohibited.

## Surge Conditions

Surge may be considered when request volume significantly exceeds available driver supply, completion rate falls below 85%, or airport queue conditions indicate insufficient supply.

Every surge adjustment must have a documented op

Source: lax_pricing.md
# LAX Pricing 

## 9. Generate Grounded Response

The retrieved policy chunks are passed to the LLM as context.
The model is instructed to answer only from the retrieved information.
This prevents the model from inventing policy rules or operational limits.
The response also includes the source policy used for the answer.

In [24]:
import os
from dotenv import load_dotenv
from google import genai

# Load API credentials from the .env file
load_dotenv()

# Read the Gemini API key from the environment
api_key = os.getenv("GEMINI_API_KEY")

# Create the Gemini client using the API key
client = genai.Client(api_key=api_key)

print("API key available:", bool(api_key))

API key available: True


In [25]:
for gemini_model in client.models.list():
    # Show models that support text generation
    if "generateContent" in gemini_model.supported_actions:
        print(gemini_model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [26]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

# Display the generated grounded answer
print(response.text)

Answer: The maximum permitted surge multiplier at SFO is 1.5x.
Source: sfo_pricing.md


In [27]:
def ask_policy(question, top_k=3):
    # Retrieve the most relevant policy chunks for the question
    results = search_policy(question, top_k)

    # Combine the retrieved chunks into context for the LLM
    context = build_context(results)

    # Build the grounded prompt using the retrieved context
    prompt = build_rag_prompt(question, context)

    # Generate the answer using the working Gemini model
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    # Keep track of which policy documents were retrieved
    sources = [result["source"] for result in results]

    return response.text, sources

In [28]:
question = "What is the maximum surge multiplier allowed at SFO?"

# Run the complete RAG pipeline
answer, sources = ask_policy(question)

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Answer:
Answer: The maximum permitted surge multiplier at SFO is 1.5x.
Source: sfo_pricing.md

Sources:
- sfo_pricing.md
- lax_pricing.md
- jfk_pricing.md


## 10. Test Questions


In [29]:
test_questions = [
    "What is the maximum surge multiplier allowed at SFO?",
    "Can drivers bypass the airport queue at LAX?",
    "What approval is required for high surge at JFK?",
    "What should be investigated when completion rate drops at SFO?",
    "When does the late-night airport policy apply?"
]

for question in test_questions:
    # Run the complete RAG pipeline
    answer, sources = ask_policy(question)

    print("=" * 80)
    print("Question:", question)
    print("\nAnswer:", answer)
    print("\nSources:")

    for source in sources:
        print("-", source)

Question: What is the maximum surge multiplier allowed at SFO?

Answer: Answer: The maximum permitted surge multiplier at SFO is 1.5x.
Source: sfo_pricing.md

Sources:
- sfo_pricing.md
- lax_pricing.md
- jfk_pricing.md
Question: Can drivers bypass the airport queue at LAX?

Answer: Answer: No, drivers must not intentionally bypass the designated airport staging queue.
Source: lax_driver_policy.md

Sources:
- lax_driver_policy.md
- lax_operations.md
- sfo_driver_policy.md
Question: What approval is required for high surge at JFK?

Answer: Answer: Surge above 1.3x requires Operations Manager approval.
Source: jfk_pricing.md

Sources:
- jfk_pricing.md
- jfk_operations.md
- jfk_driver_policy.md
Question: What should be investigated when completion rate drops at SFO?

Answer: Answer: Operations should investigate driver availability, queue conditions, cancellation rate, and ETA.
Source: sfo_operations.md

Sources:
- sfo_operations.md
- sfo_driver_policy.md
- lax_operations.md
Question: When